# 03 — Transfer Learning MobileNetV2 (2.5D) + Modèle 61 Coupes (MIL)

**Objectif :** Améliorer la baseline CNN *from scratch* en utilisant **MobileNetV2 pré-entraîné sur ImageNet**, optimisé pour ton GPU **Intel Iris (`/XPU:0`)** et **7,4 Go de RAM**, sur **exactement le même split patient** (`artifacts_modeling/splits_binary.csv`).

### Ce qu'apporte ce notebook
1. **Entrée 2.5D `(t-1, t, t+1)`** : comme MobileNetV2 attend 3 canaux RGB et que nos IRM sont en niveaux de gris, on empile **3 coupes consécutives** dans les 3 canaux pour donner un contexte 3D gratuit.
2. **Transfer Learning en 2 phases** :
   - *Phase 1 (Warm-up)* : backbone MobileNetV2 gelé, entraînement de la tête de classification.
   - *Phase 2 (Fine-tuning)* : dégel des 35 dernières couches (hors `BatchNormalization`) avec un learning rate faible (`5e-5`).
3. **Passage aux 61 coupes d'un coup** : extraction des embeddings `(61, 128)` de MobileNetV2 puis entraînement du modèle **Conv1D + Gated Attention MIL** pour renvoyer une probabilité binaire unique `P(Demented)`.
4. **Comparaison directe** sur les mêmes patients de Test : Baseline 2D *from scratch* vs MobileNetV2 2.5D vs MobileNetV2 + 61 coupes MIL.


## 1. Configuration, Chargement du Split & Construction des Triplets 2.5D `(t-1, t, t+1)`


In [ ]:
import os
import time
import warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("/home/timba/Téléchargements/Data")
OUT_DIR = DATA_DIR / "artifacts_modeling"

IMG_H, IMG_W = 128, 256  # Ratio exact 2:1 (248x496), léger pour MobileNetV2 sur Intel Iris
N_SLICES = 61
BATCH_SIZE = 32
STEPS_PER_EPOCH = 100    # ~3200 triplets par époque
EPOCHS_WARMUP = 4
EPOCHS_FINETUNE = 6

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DEVICE = "/XPU:0" if tf.config.list_physical_devices("XPU") else ("/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0")
print(f"TensorFlow : {tf.__version__} | Périphérique : {DEVICE}")

# Chargement du split strict par patient (identique aux notebooks 01 et 02)
df = pd.read_csv(OUT_DIR / "splits_binary.csv").sort_values(["volume_id", "slice_idx"]).reset_index(drop=True)

# Construction des chemins (t-1, t, t+1) au sein de chaque volume MPR (clamp aux bords 100 et 160)
df["path_prev"] = df.groupby("volume_id")["filepath"].shift(1).fillna(df["filepath"])
df["path_next"] = df.groupby("volume_id")["filepath"].shift(-1).fillna(df["filepath"])

# Mélange complet de train_df AVANT tf.data (corrige le biais de shuffle sur volumes triés)
train_df = df[df["split"] == "train"].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
val_fast_df = val_df[val_df["slice_idx"] % 3 == 0].reset_index(drop=True)

# Poids de classe
n_neg, n_pos = (train_df["target"] == 0).sum(), (train_df["target"] == 1).sum()
class_weight = {0: len(train_df) / (2.0 * n_neg), 1: len(train_df) / (2.0 * n_pos)}

print(f"Train : {len(train_df):,} | Val (fast) : {len(val_fast_df):,} | Val (full) : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Class weights : {{0: {class_weight[0]:.3f}, 1: {class_weight[1]:.3f}}}")


## 2. Pipeline `tf.data` 2.5D `(t-1, t, t+1)`

Chaque échantillon charge 3 coupes voisines en niveaux de gris, les empile en `(128, 256, 3)` et applique `mobilenet_v2.preprocess_input` (mise à l'échelle dans `[-1, 1]`). L'augmentation s'exécute côté CPU dans `tf.data` pour une compatibilité totale avec Intel ITEX (`/XPU:0`).


In [ ]:
def _read_gray(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=1)
    img = tf.image.resize(img, [IMG_H, IMG_W], method="bilinear")
    return img  # float32 dans [0, 255]

def load_triplet(p_prev, p_curr, p_next, label):
    img = tf.concat([_read_gray(p_prev), _read_gray(p_curr), _read_gray(p_next)], axis=-1)
    return img, tf.cast(label, tf.float32)

def augment_triplet(img, label):
    # Augmentation 2.5D renforcée (appliquée de façon cohérente sur les 3 coupes empilées)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.resize_with_crop_or_pad(img, IMG_H + 12, IMG_W + 16)
    img = tf.image.random_crop(img, size=[IMG_H, IMG_W, 3])
    img = tf.image.random_brightness(img, max_delta=20.0)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = img + tf.random.normal(tf.shape(img), mean=0.0, stddev=4.0)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def preprocess_mnv2(img, label):
    return preprocess_input(img), label

def make_25d_dataset(sub_df, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((
        sub_df["path_prev"].values,
        sub_df["filepath"].values,
        sub_df["path_next"].values,
        sub_df["target"].values,
    ))
    if training:
        ds = ds.shuffle(buffer_size=len(sub_df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.repeat()
    ds = ds.map(load_triplet, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment_triplet, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(preprocess_mnv2, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_25d_dataset(train_df, training=True)
val_fast_ds = make_25d_dataset(val_fast_df, training=False)
val_full_ds = make_25d_dataset(val_df, training=False, batch_size=64)
test_full_ds = make_25d_dataset(test_df, training=False, batch_size=64)

xb, yb = next(iter(train_ds))
print(f"Batch 2.5D : X={xb.shape}, y={yb.shape}, min={xb.numpy().min():.2f}, max={xb.numpy().max():.2f}")


## 3. Modèle MobileNetV2 (ImageNet) & Entraînement en 2 Phases

- **Phase 1 (Warm-up)** : backbone MobileNetV2 gelé (`trainable = False`), `lr = 1e-3`.
- **Phase 2 (Fine-tuning)** : dégel des **35 dernières couches** de MobileNetV2 (en gardant les `BatchNormalization` gelées pour préserver les statistiques ImageNet), `lr = 5e-5`.


In [ ]:
def build_mobilenetv2_25d(input_shape=(IMG_H, IMG_W, 3)):
    reg = keras.regularizers.l2(1e-3)
    base = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
    )
    base.trainable = False  # Phase 1 : gelé

    inp = keras.Input(shape=input_shape, name="mri_triplet_25d")
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.45, name="drop_1")(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=reg, name="slice_embedding")(x)
    x = layers.Dropout(0.50, name="drop_2")(x)
    out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg, name="prob_demented")(x)
    return keras.Model(inp, out, name="MobileNetV2_25D"), base

with tf.device(DEVICE):
    mnv2_model, base_backbone = build_mobilenetv2_25d()
    mnv2_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[
            keras.metrics.AUC(name="roc_auc", curve="ROC"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )

mnv2_model.summary()


In [ ]:
mnv2_path = OUT_DIR / "mobilenetv2_25d.keras"

# --- PHASE 1 : Warm-up (backbone gelé) ---
print("🚀 Phase 1 : Warm-up (backbone MobileNetV2 gelé)...")
t0 = time.time()
with tf.device(DEVICE):
    hist1 = mnv2_model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS_WARMUP,
        validation_data=val_fast_ds,
        class_weight=class_weight,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(mnv2_path), monitor="val_roc_auc", mode="max", save_best_only=True, verbose=1)
        ],
        verbose=2,
    )

# --- PHASE 2 : Fine-tuning doux (15 dernières couches dégelées, hors BatchNorm, lr très faible) ---
print("\n🔓 Phase 2 : Fine-tuning doux des 15 dernières couches (hors BatchNormalization)...")
base_backbone.trainable = True
for layer in base_backbone.layers[:-15]:
    layer.trainable = False
for layer in base_backbone.layers[-15:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

with tf.device(DEVICE):
    mnv2_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[
            keras.metrics.AUC(name="roc_auc", curve="ROC"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    hist2 = mnv2_model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS_FINETUNE,
        validation_data=val_fast_ds,
        class_weight=class_weight,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(mnv2_path), monitor="val_loss", mode="min", save_best_only=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
            keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=3, restore_best_weights=True, verbose=1),
        ],
        verbose=2,
    )

print(f"\n✅ Entraînement MobileNetV2 terminé en {(time.time()-t0)/60:.1f} min -> {mnv2_path}")

# Courbes combinées Phase 1 + Phase 2
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m, title in zip(axes, ["loss", "roc_auc", "pr_auc"], ["Loss", "ROC-AUC (coupe)", "PR-AUC (coupe)"]):
    tr = hist1.history[m] + hist2.history[m]
    va = hist1.history[f"val_{m}"] + hist2.history[f"val_{m}"]
    ax.plot(tr, label="Train", lw=2)
    ax.plot(va, label="Val (fast)", lw=2)
    ax.axvline(len(hist1.history[m]) - 0.5, color="gray", linestyle="--", label="Début fine-tuning")
    ax.set_title(title)
    ax.set_xlabel("Époque")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4. Évaluation MobileNetV2 2.5D & Extraction des Embeddings `(61, 128)`


In [ ]:
best_mnv2 = keras.models.load_model(mnv2_path)

with tf.device(DEVICE):
    val_df["pred_mnv2"] = best_mnv2.predict(val_full_ds, verbose=0).ravel()
    test_df["pred_mnv2"] = best_mnv2.predict(test_full_ds, verbose=0).ravel()

val_df[["filepath", "subject_id", "volume_id", "slice_idx", "label_4cls", "target", "pred_mnv2"]].to_csv(
    OUT_DIR / "mobilenetv2_25d_val_preds.csv", index=False
)
test_df[["filepath", "subject_id", "volume_id", "slice_idx", "label_4cls", "target", "pred_mnv2"]].to_csv(
    OUT_DIR / "mobilenetv2_25d_test_preds.csv", index=False
)

# Extraction des embeddings (61, 128) sur les 1 417 volumes pour le modèle 61 coupes MIL
emb_cache = OUT_DIR / "mobilenetv2_embeddings_61x128.npz"
print("🔄 Extraction des embeddings MobileNetV2 (128 dims) sur les 86 437 coupes...")
t0 = time.time()
extractor = keras.Model(inputs=best_mnv2.input, outputs=best_mnv2.get_layer("slice_embedding").output)
ds_all = make_25d_dataset(df, training=False, batch_size=128).map(lambda x, y: x)

with tf.device(DEVICE):
    all_embs = extractor.predict(ds_all, verbose=0).astype(np.float32)

n_vols = len(df) // N_SLICES
X_vols = all_embs.reshape(n_vols, N_SLICES, -1)
vol_meta = df.groupby("volume_id", sort=False, as_index=False).agg(
    subject_id=("subject_id", "first"),
    label_4cls=("label_4cls", "first"),
    target=("target", "first"),
    split=("split", "first"),
)
np.savez_compressed(emb_cache, X_vols=X_vols)
print(f"✅ Embeddings MobileNetV2 extraits en {time.time()-t0:.1f}s -> shape = {X_vols.shape}")


## 5. Modèle 61 Coupes d'un Coup (Conv1D + Gated Attention MIL) sur Embeddings MobileNetV2


In [ ]:
train_m = (vol_meta["split"] == "train").values
val_m = (vol_meta["split"] == "val").values
test_m = (vol_meta["split"] == "test").values

X_train, y_train = X_vols[train_m], vol_meta.loc[train_m, "target"].values.astype(np.float32)
X_val, y_val = X_vols[val_m], vol_meta.loc[val_m, "target"].values.astype(np.float32)
X_test, y_test = X_vols[test_m], vol_meta.loc[test_m, "target"].values.astype(np.float32)

# Standardisation calculée uniquement sur Train
m_tr = X_train.mean(axis=(0, 1), keepdims=True)
s_tr = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
X_train = (X_train - m_tr) / s_tr
X_val = (X_val - m_tr) / s_tr
X_test = (X_test - m_tr) / s_tr

cw_vol = {0: len(y_train) / (2.0 * (y_train == 0).sum()), 1: len(y_train) / (2.0 * (y_train == 1).sum())}

class GatedAttentionPooling(layers.Layer):
    def __init__(self, attn_dim=16, l2_reg=1e-2, **kwargs):
        super().__init__(**kwargs)
        self.attn_dim = attn_dim
        self.l2_reg = l2_reg
        reg = keras.regularizers.l2(l2_reg)
        self.V = layers.Dense(attn_dim, activation="tanh", kernel_regularizer=reg)
        self.U = layers.Dense(attn_dim, activation="sigmoid", kernel_regularizer=reg)
        self.w = layers.Dense(1, kernel_regularizer=reg)

    def call(self, inputs):
        scores = self.w(self.V(inputs) * self.U(inputs))
        weights = tf.nn.softmax(scores, axis=1)
        context = tf.reduce_sum(weights * inputs, axis=1)
        return context, tf.squeeze(weights, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"attn_dim": self.attn_dim, "l2_reg": self.l2_reg})
        return cfg

reg_mil = keras.regularizers.l2(1e-2)
inp = keras.Input(shape=(N_SLICES, X_train.shape[-1]), name="mnv2_61_slices")
x = layers.Dropout(0.25, noise_shape=(None, N_SLICES, 1), name="slice_dropout")(inp)
x = layers.SpatialDropout1D(0.30, name="feat_dropout")(x)
h = layers.Conv1D(24, kernel_size=3, padding="same", activation="relu", kernel_regularizer=reg_mil)(x)
h = layers.BatchNormalization()(h)
h = layers.Dropout(0.40)(h)

context, attn_weights = GatedAttentionPooling(attn_dim=16, l2_reg=1e-2, name="gated_attention")(h)
z = layers.Dropout(0.50)(context)
out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg_mil, name="prob_demented")(z)

mil_model = keras.Model(inp, out, name="MobileNetV2_61Slices_MIL_Compact")
mil_attn_model = keras.Model(inp, [out, attn_weights])

mil_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[keras.metrics.AUC(name="roc_auc", curve="ROC"), keras.metrics.AUC(name="pr_auc", curve="PR")],
)

mil_hist = mil_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=50,
    class_weight=cw_vol,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=4, min_lr=1e-5, verbose=0),
        keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10, restore_best_weights=True, verbose=1),
    ],
    verbose=2,
)
mil_model.save(OUT_DIR / "mobilenetv2_61_mil.keras")


## 6. Comparaison Finale sur le Test Set (Niveau Patient) & Profil d'Attention


In [ ]:
def optimal_thr(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])

def eval_patient(name, y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "Modèle (Test — Patient)": name,
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "F1 (Malade)": f1_score(y_true, y_pred, zero_division=0),
        "Seuil (Val)": thr,
    }

# 1. MobileNetV2 2.5D (moyenne simple des coupes par patient)
mnv2_val_pat = val_df.groupby("subject_id", as_index=False).agg(target=("target", "first"), prob=("pred_mnv2", "mean"))
mnv2_test_pat = test_df.groupby("subject_id", as_index=False).agg(
    label_4cls=("label_4cls", "first"), target=("target", "first"), prob=("pred_mnv2", "mean")
)
thr_mnv2 = optimal_thr(mnv2_val_pat["target"], mnv2_val_pat["prob"])

# 2. MobileNetV2 + 61 coupes MIL
val_p61, _ = mil_attn_model.predict(X_val, verbose=0)
test_p61, test_attn = mil_attn_model.predict(X_test, verbose=0)
val_vol_df = vol_meta[val_m].copy().assign(prob=val_p61.ravel())
test_vol_df = vol_meta[test_m].copy().assign(prob=test_p61.ravel())

mil_val_pat = val_vol_df.groupby("subject_id", as_index=False).agg(target=("target", "first"), prob=("prob", "mean"))
mil_test_pat = test_vol_df.groupby("subject_id", as_index=False).agg(
    label_4cls=("label_4cls", "first"), target=("target", "first"), prob=("prob", "mean")
)
thr_mil = optimal_thr(mil_val_pat["target"], mil_val_pat["prob"])

rows_comp = []
roc_curves = []

# Ajout de la Baseline 2D (notebook 01) si disponible
b2d_val_file = OUT_DIR / "baseline_2d_val_preds.csv"
b2d_test_file = OUT_DIR / "baseline_2d_test_preds.csv"
if b2d_val_file.exists() and b2d_test_file.exists():
    b2d_v = pd.read_csv(b2d_val_file).groupby("subject_id", as_index=False).agg(target=("target", "first"), prob=("pred_prob", "mean"))
    b2d_t = pd.read_csv(b2d_test_file).groupby("subject_id", as_index=False).agg(target=("target", "first"), prob=("pred_prob", "mean"))
    thr_b2d = optimal_thr(b2d_v["target"], b2d_v["prob"])
    rows_comp.append(eval_patient("01 — Baseline CNN 2D (from scratch)", b2d_t["target"], b2d_t["prob"], thr_b2d))
    roc_curves.append(("Baseline 2D", b2d_t["target"], b2d_t["prob"], "#457b9d", ":"))

rows_comp.append(eval_patient("03a — MobileNetV2 2.5D (moy. coupes)", mnv2_test_pat["target"], mnv2_test_pat["prob"], thr_mnv2))
rows_comp.append(eval_patient("03b — MobileNetV2 + 61 coupes MIL", mil_test_pat["target"], mil_test_pat["prob"], thr_mil))
roc_curves.append(("MobileNetV2 2.5D", mnv2_test_pat["target"], mnv2_test_pat["prob"], "#2a9d8f", "--"))
roc_curves.append(("MobileNetV2 + 61 coupes MIL", mil_test_pat["target"], mil_test_pat["prob"], "#e63946", "-"))

display(pd.DataFrame(rows_comp).round(4))

y_pred_mil = (mil_test_pat["prob"] >= thr_mil).astype(int)
print("\nRapport détaillé — MobileNetV2 + 61 coupes MIL (Test Patients) :")
print(classification_report(mil_test_pat["target"], y_pred_mil, target_names=["Non Demented (0)", "Demented (1)"], digits=3))

# Visualisation finale : ROC comparées + Matrice de confusion + Attention 61 coupes
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for label, yt, yp, col, ls in roc_curves:
    fpr, tpr, _ = roc_curve(yt, yp)
    axes[0].plot(fpr, tpr, color=col, linestyle=ls, lw=2.2, label=f"{label} (AUC={roc_auc_score(yt, yp):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("Courbes ROC — Test Patients")
axes[0].set_xlabel("Faux Positifs")
axes[0].set_ylabel("Vrais Positifs")
axes[0].legend(fontsize=8)

cm = confusion_matrix(mil_test_pat["target"], y_pred_mil)
axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["Prédit Sain", "Prédit Malade"])
axes[1].set_yticklabels(["Vrai Sain", "Vrai Malade"])
axes[1].set_title(f"Matrice de confusion — MIL (seuil={thr_mil:.2f})")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14, fontweight="bold",
                     color="white" if cm[i, j] > cm.max()/2 else "black")

slices_axis = np.arange(100, 161)
axes[2].plot(slices_axis, test_attn[test_vol_df["target"].values == 0].mean(axis=0), label="Non Demented (0)", color="#2a9d8f", lw=2)
axes[2].plot(slices_axis, test_attn[test_vol_df["target"].values == 1].mean(axis=0), label="Demented (1)", color="#e76f51", lw=2)
axes[2].axhline(1.0 / N_SLICES, color="gray", linestyle=":", lw=1.2, label="Uniforme (1/61)")
axes[2].set_title("Poids d'attention sur les 61 coupes (100 → 160)")
axes[2].set_xlabel("Indice de coupe (slice_idx)")
axes[2].set_ylabel("Attention moyenne")
axes[2].legend(fontsize=8.5)

plt.tight_layout()
plt.show()
